# Week 2 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-02.md`](../../weeks/week-02.md) · **Given engine:** [`vacuum.py`](vacuum.py) · [`tournament.py`](tournament.py)

This notebook is the **teaching walkthrough** of the week-2 studio. It **imports the
reference code from [`solution.py`](solution.py)** — it never re-pastes it — so the
answer shown here is byte-for-byte the one the provided test grades. The correctness
guarantee is separate: `python3 studios/_verify_solutions.py week-02` runs the
unmodified `test_agents.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: put the week folder (for solution/vacuum/tournament) and the
# repo root (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-02"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import solution
from vacuum import (VacuumWorld, BrokenSensorWorld, run_in,
                    simple_reflex, model_based)
from tournament import load_configs, net_utility, tournament

CONFIGS = load_configs()
print(f"{len(CONFIGS)} start configs loaded")

## Task 1a — `goal_based` (plan to the goal, then STOP)

The bar (`test_goal_based_reaches_goal_on_every_config` + `test_goal_based_stops_when_done`):
on **all 8** configs the agent drives both rooms clean, and once the goal is reached it
returns `"NoOp"` forever instead of oscillating. It reads the *true* `loc` from the
percept (the sensor works in `VacuumWorld`) — reference code straight from `solution.py`:

In [ ]:
print(inspect.getsource(solution.goal_based))

# mirror test_goal_based_reaches_goal_on_every_config
for dirt, loc in CONFIGS:
    w = run_in(VacuumWorld(dirt, loc), solution.goal_based(), steps=20)
    assert w.dirt == [False, False], f"did not reach goal on dirt={dirt} loc={loc}: {w}"
print("reaches (both rooms clean) on all 8 configs")

# mirror test_goal_based_stops_when_done: once clean, must NoOp and not move
agent = solution.goal_based()
w = VacuumWorld((True, True), 0)
run_in(w, agent, steps=20)
assert w.dirt == [False, False]
loc_before = w.loc
for _ in range(5):
    a = agent(w.percept())
    assert a == "NoOp", f"kept acting after goal reached: {a!r}"
    w.step(a)
assert w.loc == loc_before
print("returns NoOp once the goal is reached (does not oscillate)")

## Task 1b — `utility_based` (greedy net utility, one step at a time)

The bar (`test_utility_cleans_the_room_it_starts_in`): a dirty starting room is always
cleaned, because a `Suck` repays its cost quickly. The agent has **no lookahead** — it
only ever sees the current percept.

In [ ]:
print(inspect.getsource(solution.utility_based))

# mirror test_utility_cleans_the_room_it_starts_in
for dirt, loc in CONFIGS:
    if not dirt[loc]:
        continue
    w = run_in(VacuumWorld(dirt, loc), solution.utility_based(), steps=20)
    assert w.dirt[loc] is False, f"never cleaned its own room dirt={dirt} loc={loc}: {w}"
print("always cleans the (dirty) room it starts in")

## The guarantee fails, twice — this is the payload of week 2

**Failure 1 — the reflex agent under partial observability.** With the location
sensor dead (`BrokenSensorWorld`), the memoryless `simple_reflex` agent provably
leaves a room dirty on at least one config, while `model_based` — which never reads
`loc` — still cleans both. This is the recap question turned into `test_reflex_fails_under_partial_observability`,
and it uses only the *given* engine, so it holds even before any starter code exists.
**Internal state is what buys you the guarantee.**

In [ ]:
loc0 = [(dirt, loc) for dirt, loc in CONFIGS if loc == 0]
reflex_failures = 0
for dirt, loc in loc0:
    wm = run_in(BrokenSensorWorld(dirt, loc), model_based(), steps=20)
    assert wm.dirt == [False, False], f"model_based left dirt on {dirt},{loc}: {wm}"
    wr = run_in(BrokenSensorWorld(dirt, loc), simple_reflex, steps=20)
    if wr.dirt != [False, False]:
        reflex_failures += 1
assert reflex_failures > 0
print(f"GUARANTEE FAIL: simple_reflex left a room dirty on {reflex_failures}/{len(loc0)} "
      f"broken-sensor configs; model_based cleaned all (memory vs. no memory).")

**Failure 2 — the *rational* agent that leaves a room dirty.** `test_utility_leaves_a_room_dirty`
asserts your greedy `utility_based` leaves >=1 room dirty on >=1 config. **Not a bug.**
When the current room is clean, the only move crosses to a room the agent cannot see;
a crossing cleans nothing on the step it happens, so its immediate net change is
`-move_cost` (a strict loss) versus `0` for `NoOp`. A horizon-free agent will not pay
to travel toward an unseen payoff. This is specification-following under a cost model —
the first agent that makes a decision you did not intend but cannot argue with.

In [ ]:
left_dirty = 0
for dirt, loc in CONFIGS:
    w = run_in(VacuumWorld(dirt, loc), solution.utility_based(), steps=20)
    if w.dirt != [False, False]:
        left_dirty += 1
assert left_dirty > 0
print(f"GUARANTEE FAIL (by design): utility_based rationally left a room dirty on "
      f"{left_dirty}/{len(CONFIGS)} configs — greedy, no horizon.")

## The tournament — net utility, four classical agents, 8 configs

`tournament()` recomputes the reward/cost ledger itself (it never trusts the agent).
The **intended discovery**: `utility` does not top the table — it leaves value on the
table because the `percept -> action` socket gives it no horizon. That is the honest
motivation for search and planning in weeks 3–9.

In [ ]:
totals = tournament({
    "reflex":  lambda: simple_reflex,   # bare function -> wrap in a factory
    "model":   model_based,
    "goal":    solution.goal_based,
    "utility": solution.utility_based,
})
print("\ntotals:", totals)

## Task 2 — the LLM as agent function (headless demo)

The abstraction from 1995 still holds: a language model is just another policy plugged
into the **same** `percept -> action` socket. Below we run it headless with the
deterministic `echo` backend so this notebook executes anywhere.

> **`echo` is a deterministic FAKE, not a model.** Its text must never be reported as a
> model result. For a real comparison, re-run with `backend="ollama"` or
> `backend="manual"`. The point the demo *does* make even on `echo`: an LLM agent needs
> defensive code — a parser, a fallback, and a **parse-failure log** (scorecard axis 8) —
> that the classical agents never needed.

In [ ]:
import os, re
from aicourse.llm import LLM

VALID = {"LEFT": "Left", "RIGHT": "Right", "SUCK": "Suck", "NOOP": "NoOp"}
parse_failures = []   # axis 8: log every malformed reply

def parse_action(raw):
    # survive "Suck.", "I would suck", or a paragraph of reasoning
    for tok in re.findall(r"[A-Za-z]+", raw or ""):
        if tok.upper() in VALID:
            return VALID[tok.upper()]
    parse_failures.append(raw)
    return None

PROMPT = ('You control a vacuum robot in a 2-room world. Rooms 0 and 1, each Clean or '
          'Dirty. Actions: Left, Right, Suck, NoOp.\n'
          'Current percept: location={loc}, status={status}\n'
          'Reply with exactly one action word and nothing else.')

backend = os.environ.get("AICOURSE_NB_BACKEND", "echo")   # echo = headless-safe fake
llm = LLM(backend=backend)

def llm_agent(percept):
    loc, dirty = percept
    resp = llm.complete(PROMPT.format(loc=loc, status="Dirty" if dirty else "Clean"))
    action = parse_action(resp.text)
    return action if action is not None else "NoOp"   # fallback on parse failure

w = run_in(VacuumWorld((True, True), 0), llm_agent, steps=6)
print("backend           :", backend, "(echo = FAKE, not a model)" if backend == "echo" else "")
print("world after 6 steps:", w)
print("parse failures     :", len(parse_failures),
      "<- defensive code the classical agents never needed (axis 8)")